# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and perform basic analyses on the FAIR^2 clinical dataset using the [mlcroissant](https://mlcommons.github.io/croissant/) Python library.

### Dataset Source
The dataset is described using a Croissant JSON-LD schema available at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load dataset metadata and the records themselves using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings

# Silence SettingWithCopyWarning for demonstration purposes
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# Define Croissant dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"{meta.name}: {meta.description}")
print(f"Identifier: {meta.identifier}")
print(f"Published: {meta.datePublished}")

## 2. Data Overview

Review available record sets, their fields, and the corresponding `@id` for referencing and extraction.

Note: All references below use the `@id` of entities as required.

In [ ]:
# List all record sets in the Croissant dataset (by their @id)
# This code block will print the @id, name, and fields for each record set.

record_set_ids = []
print('Available Record Sets:')
for rs in dataset.record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {getattr(rs, 'name', '(Unnamed)')}")
    print(f"  Description: {getattr(rs, 'description', '(No description)')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id} | Name: {getattr(field, 'name', '(Unnamed)')} | Data type: {getattr(field, 'dataType', '-')}")
    record_set_ids.append(rs.id)
    print()

## 3. Data Extraction

We will load the data from each available record set into a dictionary of DataFrames using their `@id` for consistent referencing.

*For this dataset, there is typically a main record set containing patient-level clinical records, plus possibly others for specific entities (e.g. treatments, pathology, etc.).*

In [ ]:
# Use record set IDs discovered in the previous cell:
dataframes = {}

for record_set_id in record_set_ids:
    # Extract all records from the record set (referencing via @id)
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet @id: {record_set_id}")
    if len(df.columns) > 0:
        print(f"  Fields: {df.columns.tolist()}")
    print()

# For demonstration, pick the main patient record set (usually has the most fields or most rows).
main_record_set_id = max(dataframes, key=lambda k: len(dataframes[k].columns)) if dataframes else None
if main_record_set_id:
    print(f"Using primary RecordSet: {main_record_set_id}")
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply basic processing: filter by a numeric field, normalize, and group by a key attribute.

*For illustration, we'll operate on age and anatomical site if present. Please adjust the field @ids for your actual data schema as listed above.*

Let's select a numeric field (e.g., `age` field with its `@id` as found above), filter ages > 50, normalize them, and group by anatomical site if available.

In [ ]:
# Replace these with actual @ids displayed previously for your dataset

df = dataframes[main_record_set_id]

# Attempt to infer likely field names/ids for demonstration
possible_age_ids = [col for col in df.columns if col.lower() in ("age", "patient_age", "年龄")]
if possible_age_ids:
    age_field_id = possible_age_ids[0]  # Use the first candidate
else:
    age_field_id = df.columns[0]  # As fallback, just use first column

print(f"Numeric field selected for analysis (as @id): {age_field_id}")

# Ensure numeric
df[age_field_id] = pd.to_numeric(df[age_field_id], errors='coerce')

threshold = 50
filtered_df = df[df[age_field_id] > threshold].copy()
print(f"Filtered records with {age_field_id} > {threshold}:")
print(filtered_df[[age_field_id]].head())

# Normalization (standard z-score for age)
filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
print(f"\nNormalized {age_field_id} for filtered records:")
print(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

# Try to select a group field (likely anatomical location/site, e.g. "anatomical_site" or similar)
possible_group_ids = [col for col in df.columns if 'site' in col.lower() or 'location' in col.lower() or 'anatomical' in col.lower()]
if possible_group_ids:
    group_field_id = possible_group_ids[0]
    print(f"\nGrouping by: {group_field_id}")
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[age_field_id].mean().to_frame(name=f"mean_{age_field_id}")
        print(f"\nMean {age_field_id} by {group_field_id}:")
        print(grouped_df.head())
else:
    print("No suitable group field for grouping found in columns.")

## 5. Visualization

Visualize distribution of the selected numeric field and the grouped means if groups were formed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# Visualize age distribution (histogram)

if age_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[age_field_id].dropna(), bins=15, color='royalblue', edgecolor='black')
    plt.title('Distribution of Age')
    plt.xlabel('Age')
    plt.ylabel('Count')
    plt.show()

if 'group_field_id' in locals() and group_field_id in filtered_df.columns:
    ordered = grouped_df.reset_index().sort_values(f"mean_{age_field_id}", ascending=False)
    plt.figure(figsize=(8,4))
    sns.barplot(x=group_field_id, y=f"mean_{age_field_id}", data=ordered, palette='mako')
    plt.xticks(rotation=45, ha='right')
    plt.title(f'Mean {age_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {age_field_id}')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- This notebook demonstrated programmatic access to the FAIR^2 clinical dataset through its Croissant schema using only stable `@id` references.
- Key fields, including numeric attributes such as age, were analyzed and grouped by anatomical site (or the closest available group attribute).
- Using mlcroissant, you can flexibly extract, transform, and analyze any Croissant-compliant dataset in a reproducible and FAIR manner.

**Next steps:** further analysis, predictive modeling, or integration with external clinical data can follow this template.